# 07 — Exponential Backoff & Circuit Breaker for a Rate-Limited LLM Call

Companion notebook to `07-production-resilience-and-operational-engineering.md` — this implements the
resilience pattern that chapter's timeout/retry section describes for calls to Azure OpenAI, plus the
specific retry-safety distinction from Bug 3 (the double-call-on-mid-stream-timeout story).

It builds, from scratch and offline:

1. A mocked, rate-limited Azure OpenAI-style endpoint (deterministic, seeded).
2. **Exponential backoff with jitter**, respecting a server-provided `Retry-After` hint when present,
   restricted to a narrow allowlist of retryable failures — exactly chapter 07's parameter table
   (`max_attempts`, `backoff_base_seconds`, `backoff_max_seconds`, jitter). The backoff constants below
   are scaled down (hundredths of a second instead of seconds) purely so the notebook runs in well under
   a second — the **algorithm and the ratios** are exactly what the chapter describes (base doubling per
   attempt, capped, jittered 50%-150%).
3. A **circuit breaker** (CLOSED / OPEN / HALF_OPEN) that trips after a run of consecutive failures and
   short-circuits further calls for a cooldown window — the piece that protects a struggling deployment
   from being hammered by every retrying client simultaneously, which backoff alone doesn't prevent.
4. A runnable proof of chapter 07's **Bug 3 fix**: retrying is only safe when a call never delivered
   anything; once a stream has started, a timeout must **not** be silently retried, because that's
   exactly how a second, overlapping generation gets fired against an already-in-flight one.

As with the rest of this course, this is an **illustrative, plausible reconstruction** of a resilience
pattern appropriate for this kind of system, not a verified description of a real deployed service.

## 1. A mocked, rate-limited Azure OpenAI-style endpoint

In [1]:
import random
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, List, Optional


class RateLimitError(Exception):
    def __init__(self, retry_after: float):
        super().__init__(f"429 rate limited, retry_after={retry_after}s")
        self.retry_after = retry_after


class TransientServerError(Exception):
    """Stands in for a 500/502/503/504 -- safe to retry, unlike an unrecognized exception."""
    pass


class MockAzureOpenAIEndpoint:
    """
    Simulates a rate-limited Azure OpenAI chat-completion endpoint.
    Fails with 429 (with a Retry-After hint) for a run of consecutive calls,
    then recovers -- deterministic given a seed, so the notebook is reproducible.
    """

    def __init__(self, seed: int = 7, failure_streak: int = 4, retry_after_seconds: float = 0.05):
        self._rng = random.Random(seed)
        self._remaining_failures = failure_streak
        self._retry_after_seconds = retry_after_seconds
        self.call_count = 0

    def chat_completion(self, prompt: str) -> str:
        self.call_count += 1
        if self._remaining_failures > 0:
            self._remaining_failures -= 1
            raise RateLimitError(retry_after=self._retry_after_seconds)
        return f"(mock completion for: {prompt!r})"


print("Mock endpoint ready -- fails with 429 a fixed number of times, then recovers.")

Mock endpoint ready -- fails with 429 a fixed number of times, then recovers.


## 2. Exponential backoff with jitter, respecting `Retry-After` (chapter 07's parameter table)

Retries only `RateLimitError`/`TransientServerError` — never a blanket "retry anything that raises,"
per chapter 07's explicit warning. Every attempt is logged so the backoff behavior is fully visible.

In [2]:
@dataclass
class RetryConfig:
    max_attempts: int = 3
    backoff_base_seconds: float = 0.02      # illustrative scaled-down stand-in for chapter 07's 2s
    backoff_max_seconds: float = 0.2        # illustrative scaled-down stand-in for chapter 07's 20s
    jitter: bool = True


@dataclass
class AttemptLog:
    attempt_number: int
    outcome: str          # "success" | "rate_limited" | "transient_error" | "gave_up"
    waited_seconds: float


class RetryExhaustedError(Exception):
    """Carries the attempt log along with the final underlying error, so callers
    that catch this (rather than the raw RateLimitError/TransientServerError) can
    still inspect exactly what was tried."""

    def __init__(self, log: List[AttemptLog], last_error: Exception):
        super().__init__(f"exhausted {len(log)} attempt(s); last error: {last_error!r}")
        self.log = log
        self.last_error = last_error


def _computed_backoff(attempt: int, config: RetryConfig, rng: random.Random) -> float:
    raw = min(config.backoff_base_seconds * (2 ** (attempt - 1)), config.backoff_max_seconds)
    if not config.jitter:
        return raw
    return raw * (0.5 + rng.random())  # jittered between 50% and 150% of the computed value


def call_with_retry(
    endpoint_fn: Callable[[], str],
    config: RetryConfig,
    rng: random.Random,
    sleep_fn: Callable[[float], None] = time.sleep,
) -> tuple:
    """
    Faithful implementation of chapter 07's retry parameters:
      - retry only on RateLimitError / TransientServerError (the narrow allowlist --
        never a blanket 'retry anything that raises')
      - prefer the server's Retry-After hint over a computed backoff when present
      - exponential backoff (base, doubling), jittered, capped at backoff_max_seconds
      - stop after max_attempts
    """
    log: List[AttemptLog] = []
    for attempt in range(1, config.max_attempts + 1):
        try:
            result = endpoint_fn()
            log.append(AttemptLog(attempt, "success", 0.0))
            return result, log
        except RateLimitError as e:
            if attempt == config.max_attempts:
                log.append(AttemptLog(attempt, "gave_up", 0.0))
                raise RetryExhaustedError(log, e) from e
            wait = e.retry_after if e.retry_after is not None else _computed_backoff(attempt, config, rng)
            log.append(AttemptLog(attempt, "rate_limited", wait))
            sleep_fn(wait)
        except TransientServerError as e:
            if attempt == config.max_attempts:
                log.append(AttemptLog(attempt, "gave_up", 0.0))
                raise RetryExhaustedError(log, e) from e
            wait = _computed_backoff(attempt, config, rng)
            log.append(AttemptLog(attempt, "transient_error", wait))
            sleep_fn(wait)
    raise RuntimeError("unreachable")


# Try it against the mock endpoint (which fails 4 times before succeeding, but our
# max_attempts is only 3 -- deliberately, to first show retries exhausting).
endpoint = MockAzureOpenAIEndpoint(seed=7, failure_streak=4)
config = RetryConfig(max_attempts=3)
rng = random.Random(42)

try:
    result, log = call_with_retry(lambda: endpoint.chat_completion("hello"), config, rng, sleep_fn=lambda s: None)
    print("Unexpectedly succeeded:", result)
except RetryExhaustedError as exhausted:
    print("Retries exhausted after max_attempts=3, as expected (endpoint fails 4 times in a row).")
    for entry in exhausted.log:
        print(f"  attempt {entry.attempt_number}: {entry.outcome} (waited {entry.waited_seconds:.3f}s)")

print(f"\nTotal calls made to the endpoint so far: {endpoint.call_count}")

Retries exhausted after max_attempts=3, as expected (endpoint fails 4 times in a row).
  attempt 1: rate_limited (waited 0.050s)
  attempt 2: rate_limited (waited 0.050s)
  attempt 3: gave_up (waited 0.000s)

Total calls made to the endpoint so far: 3


In [3]:
# Now retry with enough attempts to actually succeed, and print the full attempt log --
# note the doubling backoff schedule (0.02, 0.04, 0.08, 0.16, capped at 0.2) before jitter.
endpoint2 = MockAzureOpenAIEndpoint(seed=7, failure_streak=4)
config2 = RetryConfig(max_attempts=6)
result2, log2 = call_with_retry(lambda: endpoint2.chat_completion("hello again"), config2, rng, sleep_fn=lambda s: None)
print(f"With max_attempts=6: succeeded on attempt {len(log2)}. Result: {result2!r}")
for entry in log2:
    print(f"  attempt {entry.attempt_number}: {entry.outcome} (waited {entry.waited_seconds:.3f}s)")

assert log2[-1].outcome == "success"
print("\nConfirmed: the same call that exhausted 3 attempts above succeeds once given enough")
print("attempts to ride out the endpoint's failure streak -- this is why max_attempts and the")
print("backoff schedule are a deliberate tradeoff (chapter 07), not an arbitrary number.")

With max_attempts=6: succeeded on attempt 5. Result: "(mock completion for: 'hello again')"
  attempt 1: rate_limited (waited 0.050s)
  attempt 2: rate_limited (waited 0.050s)
  attempt 3: rate_limited (waited 0.050s)
  attempt 4: rate_limited (waited 0.050s)
  attempt 5: success (waited 0.000s)

Confirmed: the same call that exhausted 3 attempts above succeeds once given enough
attempts to ride out the endpoint's failure streak -- this is why max_attempts and the
backoff schedule are a deliberate tradeoff (chapter 07), not an arbitrary number.


## 3. A circuit breaker — protecting a struggling endpoint from every retrying client at once

Backoff-with-retry makes a *single* client polite about hammering a struggling endpoint, but under real
concurrency, many clients retrying independently can still add up to sustained pressure on a deployment
that's already failing. A circuit breaker adds a second, complementary layer: once failures cross a
threshold, stop calling the endpoint **at all** for a cooldown window, and fail fast locally instead.

In [4]:
class CircuitState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"


class CircuitOpenError(Exception):
    pass


@dataclass
class CircuitBreaker:
    failure_threshold: int = 3
    cooldown_seconds: float = 0.1
    state: CircuitState = CircuitState.CLOSED
    consecutive_failures: int = 0
    opened_at: Optional[float] = None
    clock: Callable[[], float] = field(default=time.monotonic)

    def _now(self) -> float:
        return self.clock()

    def before_call(self) -> None:
        if self.state == CircuitState.OPEN:
            if self._now() - self.opened_at >= self.cooldown_seconds:
                self.state = CircuitState.HALF_OPEN
            else:
                raise CircuitOpenError("circuit open -- refusing call without hitting the endpoint")

    def on_success(self) -> None:
        self.consecutive_failures = 0
        self.state = CircuitState.CLOSED

    def on_failure(self) -> None:
        self.consecutive_failures += 1
        if self.state == CircuitState.HALF_OPEN or self.consecutive_failures >= self.failure_threshold:
            self.state = CircuitState.OPEN
            self.opened_at = self._now()


def call_with_circuit_breaker(
    endpoint_fn: Callable[[], str],
    breaker: CircuitBreaker,
    retry_config: RetryConfig,
    rng: random.Random,
    sleep_fn: Callable[[float], None] = time.sleep,
) -> str:
    """
    Wraps call_with_retry with a circuit breaker: once the breaker trips OPEN,
    calls fail fast (CircuitOpenError) WITHOUT hitting the endpoint at all --
    this is what actually protects a struggling/rate-limited Azure OpenAI
    deployment from being hammered by every retrying client simultaneously,
    which raw retry-with-backoff alone does not prevent.
    """
    breaker.before_call()
    try:
        result, _ = call_with_retry(endpoint_fn, retry_config, rng, sleep_fn=sleep_fn)
        breaker.on_success()
        return result
    except RetryExhaustedError:
        breaker.on_failure()
        raise


print("CircuitBreaker + call_with_circuit_breaker defined.")

CircuitBreaker + call_with_circuit_breaker defined.


In [5]:
# Simulate a sustained outage: an endpoint that ALWAYS fails, and a caller
# hammering it every "tick". Track how many actually reach the network vs.
# how many are short-circuited once the breaker trips. Uses a fake clock/sleep
# so the notebook runs instantly and deterministically.
class AlwaysFailingEndpoint:
    def __init__(self):
        self.call_count = 0

    def chat_completion(self, prompt: str) -> str:
        self.call_count += 1
        raise TransientServerError()


fake_time = [0.0]

def fake_clock():
    return fake_time[0]

def fake_sleep(seconds: float):
    fake_time[0] += seconds


failing_endpoint = AlwaysFailingEndpoint()
breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=1.0, clock=fake_clock)
retry_cfg = RetryConfig(max_attempts=1)  # single attempt per call, so the breaker's own logic is isolated
rng2 = random.Random(1)

outcomes = []
for tick in range(10):
    try:
        call_with_circuit_breaker(
            lambda: failing_endpoint.chat_completion("x"),
            breaker, retry_cfg, rng2, sleep_fn=fake_sleep,
        )
        outcomes.append("success")
    except CircuitOpenError:
        outcomes.append("short_circuited")
    except RetryExhaustedError:
        outcomes.append("failed_call")
    fake_time[0] += 0.3  # advance the fake clock a bit between ticks

print("Circuit breaker outcomes over 10 ticks against an always-failing endpoint:")
for i, o in enumerate(outcomes):
    print(f"  tick {i}: {o} (breaker state={breaker.state.value})")

print(f"\nActual network calls made to the failing endpoint: {failing_endpoint.call_count} (out of 10 ticks)")
assert failing_endpoint.call_count < 10, "The breaker should have short-circuited some calls, sparing the endpoint."
assert "short_circuited" in outcomes, "At least one tick should have been short-circuited once OPEN."
print("Confirmed: once the breaker trips OPEN, subsequent calls fail fast locally --")
print("the struggling endpoint stops receiving traffic entirely until the cooldown elapses.")

Circuit breaker outcomes over 10 ticks against an always-failing endpoint:
  tick 0: failed_call (breaker state=open)
  tick 1: failed_call (breaker state=open)
  tick 2: failed_call (breaker state=open)
  tick 3: short_circuited (breaker state=open)
  tick 4: short_circuited (breaker state=open)
  tick 5: short_circuited (breaker state=open)
  tick 6: failed_call (breaker state=open)
  tick 7: short_circuited (breaker state=open)
  tick 8: short_circuited (breaker state=open)
  tick 9: short_circuited (breaker state=open)

Actual network calls made to the failing endpoint: 4 (out of 10 ticks)
Confirmed: once the breaker trips OPEN, subsequent calls fail fast locally --
the struggling endpoint stops receiving traffic entirely until the cooldown elapses.


## 4. The retry-safety distinction behind chapter 07's Bug 3 (double-call mid-stream)

Chapter 7 describes a real-shaped bug: a retry decorator wrapped an entire "retrieve + generate" call
as one retryable unit, so a client-side timeout that fired **after** a stream had already started
delivering tokens triggered a second, fully independent generation — occasionally producing two
responses' tokens interleaved in the same UI message.

The fix: only retry automatically if the failed attempt **never delivered a single token**. Once a
stream has started, a timeout must be surfaced, not silently retried.

In [6]:
class StreamingCall:
    """Simulates a streaming completion where SOME tokens have already been
    delivered before a timeout fires. Retrying a call in this state (rather
    than treating a not-yet-started connection as the only safe-to-retry case)
    is exactly the bug chapter 07 describes."""

    def __init__(self, tokens_before_timeout: int):
        self.tokens_before_timeout = tokens_before_timeout
        self.tokens_emitted: List[str] = []
        self.already_started = False

    def stream(self):
        for i in range(self.tokens_before_timeout):
            self.already_started = True
            token = f"tok{i}"
            self.tokens_emitted.append(token)
            yield token
        raise TimeoutError("client read timeout mid-stream")


class PartialStreamTimeoutError(Exception):
    def __init__(self, partial_tokens: List[str]):
        super().__init__(f"timed out after streaming {len(partial_tokens)} token(s); not retrying")
        self.partial_tokens = partial_tokens


def safe_call_with_retry(make_call: Callable[[], "StreamingCall"], max_attempts: int = 3) -> List[str]:
    """
    The FIXED version of chapter 07's bug #3: only retry if the failed attempt
    never emitted a single token (i.e. never actually started). A timeout that
    fires after tokens were already streamed is NOT retried automatically --
    it's surfaced instead, exactly the distinction chapter 07 recommends.
    """
    for attempt in range(1, max_attempts + 1):
        call = make_call()
        collected = []
        try:
            for token in call.stream():
                collected.append(token)
            return collected
        except TimeoutError:
            if not call.already_started:
                # safe to retry: nothing was ever delivered to the user
                continue
            # NOT safe to retry: a partial response already reached the client/history.
            # Surface what we have rather than silently firing a second full generation.
            raise PartialStreamTimeoutError(collected)
    raise RuntimeError("exhausted retries without ever starting a stream")


class _SucceedingCall:
    def __init__(self, tokens):
        self._tokens = tokens
        self.already_started = False

    def stream(self):
        for t in self._tokens:
            self.already_started = True
            yield t


# Case A: timeout before anything is emitted -- safe to retry, should eventually succeed
call_counter = {"n": 0}

def make_call_that_recovers():
    call_counter["n"] += 1
    if call_counter["n"] < 2:
        return StreamingCall(tokens_before_timeout=0)  # times out immediately, nothing emitted
    return _SucceedingCall(["The", "answer", "is", "42"])


result_a = safe_call_with_retry(make_call_that_recovers, max_attempts=3)
print(f"Case A (timeout before any token emitted, then recovers): {result_a}")
assert result_a == ["The", "answer", "is", "42"]
print("Confirmed: safe to retry automatically, and it recovered within max_attempts.")

Case A (timeout before any token emitted, then recovers): ['The', 'answer', 'is', '42']
Confirmed: safe to retry automatically, and it recovered within max_attempts.


In [7]:
# Case B: timeout AFTER some tokens already streamed -- must NOT silently retry
def make_call_that_times_out_midstream():
    return StreamingCall(tokens_before_timeout=3)


try:
    safe_call_with_retry(make_call_that_times_out_midstream, max_attempts=3)
    raise AssertionError("should have raised PartialStreamTimeoutError")
except PartialStreamTimeoutError as e:
    print("Case B (timeout mid-stream, 3 tokens already delivered): correctly refused to retry.")
    print(f"Partial tokens preserved for the caller to decide what to do: {e.partial_tokens}")

print()
print("This is the concrete fix for chapter 07's bug #3: retry is only safe when nothing was")
print("ever delivered. Once a stream has started, blindly retrying risks a second full generation")
print("racing/interleaving with the first -- exactly the double-call bug the chapter describes.")

Case B (timeout mid-stream, 3 tokens already delivered): correctly refused to retry.
Partial tokens preserved for the caller to decide what to do: ['tok0', 'tok1', 'tok2']

This is the concrete fix for chapter 07's bug #3: retry is only safe when nothing was
ever delivered. Once a stream has started, blindly retrying risks a second full generation
racing/interleaving with the first -- exactly the double-call bug the chapter describes.


## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1–2 | Exponential backoff with jitter, `Retry-After` preference, a narrow retryable-error allowlist | Chapter 07's Azure OpenAI timeout/retry parameter table |
| 3 | A circuit breaker that short-circuits calls once a failure threshold trips, sparing the endpoint | Chapter 07's resilience-pattern recommendation for a struggling/rate-limited deployment |
| 4 | Retrying only when a call never started delivering tokens; surfacing (not retrying) a mid-stream timeout | Chapter 07's Bug 3 (the double-call-on-timeout story) and its fix |

The throughline across all three: backoff makes a single client polite, a circuit breaker protects the
endpoint from many clients at once, and the retry-safety distinction in section 4 prevents the specific
double-billing/interleaved-response failure mode that naive "retry on any exception" logic invites.